In [259]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torchbnn as bnn
from tqdm.auto import tqdm




# Установка случайного сида для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [260]:
df = pd.read_csv('../data/data.csv')
df

,"Sigma, Mpa",T.K,t.h,Th,C,Cr,Co,Mo,W,Al,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni
0,241.316495,1144.2600,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,NaN,99.500
1,241.316495,1144.2600,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.75,NaN,NaN,NaN,99.250
2,241.316495,1144.2600,68.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.05,99.450
3,241.316495,1144.2600,40.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.20,99.300
4,241.316495,1144.2600,32.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.50,99.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,922.0389,1183.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3442,1103.161120,866.4833,25554.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3443,241.316495,1199.8170,183.2,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750
3444,241.316495,1199.8170,153.0,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750


In [261]:
mass = '232,03806	12,011	51,9961	58,933194	95,95	183,84	26,9815385	47,867	92,90637	10,81	55,845	88,90584	91,224	180,94788	186,207	101,07	50,9415	140,116	138,90547	32,06	28,085	54,938044	24,305	30,973761998	178,49	107,8682	63,546	208,9804	207,2	192,22	72,63	69,723	58,6934'.replace(',', '.').split()
element = 'Th	C	Cr	Co	Mo	W	Al	Ti	Nb	B	Fe	Y	Zr	Ta	Re	Ru	V	Ce	La	S	Si	Mn	Mg	P	Hf	Ag	Cu	Bi	Pb	Ir	Ge	Ga	Ni'.split()
atom_mass = dict(zip(element, mass))

In [262]:
atom_mass['Ni']

'58.6934'

In [263]:
for elem in df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list():
    df[elem] = df[elem] / float(atom_mass[elem])
df['sum'] = df[df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list()].sum(axis=1, skipna=True)
df['PLM'] = df['T.K'] * (20 + np.log10(df['t.h'])) * 1e-5

cols = df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns
df.loc[:, cols] = df.loc[:, cols].div(df['sum'], axis=0)
df.loc[:, cols] = df.loc[:, cols].div(df['Ni'], axis=0)



df = df.fillna(0)

df = df.drop(columns=['T.K', 't.h', 'Ni', 'sum'])

df

,"Sigma, Mpa",Th,C,Cr,Co,Mo,W,Al,Ti,Nb,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,PLM
0,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001652,0.000000,0.0,0.000000,0.139405
1,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.002485,0.000000,0.0,0.000000,0.149239
2,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001653,0.000000,0.0,0.000423,0.147465
3,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001656,0.000000,0.0,0.001695,0.146154
4,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001661,0.000000,0.0,0.004252,0.145925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,0.112115,0.020497,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.368295
3442,1103.161120,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,0.112115,0.020497,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.366118
3443,241.316495,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,0.112870,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.262391
3444,241.316495,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,0.112870,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.261469


In [264]:
def y_from_sigma(sigma_mpa):
    # y = -log10(sigma)
    return -np.log10(sigma_mpa)

def sigma_from_y(y):
    # sigma = 10^(-y)
    return 10 ** (-y)

def calculate_mse(y_pred, y_true):
    """Вычисление MSE (Mean Squared Error)"""
    return torch.mean((y_pred - y_true) ** 2).item()

def calculate_rmse_absolute(sigma_pred, sigma_real):
    """
    Вычисление RMSE для физических значений σ (MPa)
    Корень из среднего квадрата абсолютных ошибок
    """
    mse = torch.mean((sigma_pred - sigma_real) ** 2).item()
    return np.sqrt(mse)

def calculate_rmse_relative(sigma_pred, sigma_real):
    """
    Вычисление RMSE согласно формуле (6) из статьи.
    Это относительная ошибка (Root Mean Square Relative Error)
    """
    epsilon = 1e-8
    relative_error = (sigma_pred - sigma_real) / (sigma_real.abs() + epsilon)
    mse_relative = torch.mean(relative_error ** 2).item()
    return np.sqrt(mse_relative)

def calculate_metrics_np(pred, real):
    eps = 1e-8
    # Абсолютная RMSE
    rmse_abs = np.sqrt(np.mean((pred - real) ** 2))
    
    # Относительная RMSE (Формула 6 из статьи)
    rel_err = (pred - real) / (real + eps)
    rmse_rel = np.sqrt(np.mean(rel_err ** 2))
    
    # MAPE
    mape = np.mean(np.abs(rel_err)) * 100
    
    # R²
    ss_res = np.sum((real - pred) ** 2)
    ss_tot = np.sum((real - np.mean(real)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    return rmse_abs, rmse_rel, mape, r2

In [265]:
target_col = 'Sigma, Mpa'

y = np.asarray(y_from_sigma(df[target_col]), dtype=np.float32).reshape(-1, 1).astype("float32")
X = df.copy().drop(columns=[target_col]).to_numpy().astype("float32")
X, y

(array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.13940506],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.1492392 ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.00042323,
         0.14746492],
        ...,
        [0.        , 0.01472125, 0.18892115, ..., 0.        , 0.        ,
         0.2623908 ],
        [0.        , 0.01472125, 0.18892115, ..., 0.        , 0.        ,
         0.26146874],
        [0.        , 0.01472125, 0.18892115, ..., 0.        , 0.        ,
         0.2588929 ]], shape=(3446, 27), dtype=float32),
 array([[-2.382587],
        [-2.382587],
        [-2.382587],
        ...,
        [-2.382587],
        [-2.382587],
        [-2.440579]], shape=(3446, 1), dtype=float32))

In [266]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    shuffle=True
)


## brann

In [267]:
class BRANN(nn.Module):
    """
    Bayesian Regularized Artificial Neural Network
    """
    def __init__(self, input_size=24, hidden_size=13):
        super(BRANN, self).__init__()
        self.hidden = nn.Linear(input_size, hidden_size)
        self.activation = nn.Tanh()
        self.output = nn.Linear(hidden_size, 1)
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Инициализация весов малыми значениями"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=0.01)  # Меньшие веса для лучшей сходимости
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.hidden(x)
        x = self.activation(x)
        x = self.output(x)
        return x
    
    def get_all_weights(self):
        """Получить все веса сети как один вектор"""
        weights = []
        for param in self.parameters():
            weights.append(param.view(-1))
        return torch.cat(weights)
    
    def compute_ed(self, X, y):
        """E_D - ошибка данных: Σ[y_i - f(X_i)]²"""
        predictions = self.forward(X)
        ed = torch.sum((predictions - y) ** 2)
        return ed
    
    def compute_ew(self):
        """E_W - ошибка весов: Σw_j²"""
        ew = 0.0
        for param in self.parameters():
            ew += torch.sum(param ** 2)
        return ew
    
    def compute_regularized_loss(self, ed, ew, alpha, beta):
        """S(w) = β·E_D + α·E_W (формула 17)"""
        return beta * ed + alpha * ew

In [268]:
def compute_effective_parameters(model, alpha, beta):
    """
    Вычисление эффективного количества параметров γ (формула 28)
    γ = N_W - α·trace(G^(-1))
    """
    w = model.get_all_weights()
    n_weights = len(w)
    
    # Приближенное вычисление через диагональ гессиана
    hessian_diag = []
    for param in model.parameters():
        # Для квадратичной функции: H_ii ≈ 2β + 2α
        hessian_diag.append(torch.ones_like(param) * (2 * beta + 2 * alpha))
    
    hessian_diag_flat = torch.cat([h.view(-1) for h in hessian_diag])
    trace_inv = torch.sum(1.0 / (hessian_diag_flat + 1e-8))
    
    gamma = n_weights - alpha * trace_inv.item()
    gamma = max(0, min(gamma, n_weights))
    
    return gamma, n_weights

## train brann

In [269]:
def train_single_brann(X_train, y_train, X_val, y_val, 
                       input_size=27, max_iterations=10, epochs_per_iteration=10,
                       alpha_init=10, beta_init=10, verbose=True):
    """
    Улучшенное обучение BRANN с защитой от расходимости α
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = BRANN(input_size).to(device)
    
    X_train_t = torch.FloatTensor(X_train).to(device)
    y_train_t = torch.FloatTensor(y_train).view(-1, 1).to(device)
    X_val_t = torch.FloatTensor(X_val).to(device)
    y_val_t = torch.FloatTensor(y_val).view(-1, 1).to(device)
    
    alpha = alpha_init
    beta = beta_init
    optimizer = optim.Adam(model.parameters(), lr=0.1)  # Увеличенный LR
    
    best_model_state = None
    best_mse_val = float('inf')
    stopped_early = False
    n_data = len(X_train)
    
    # Для ограничения резких изменений
    alpha_prev, beta_prev = alpha, beta

    
    history = {'Iter':[0]
               ,'alpha':[alpha_init]
               ,'beta':[beta_init]
               ,'gamma': [0]
               ,'MSE_tr':[0]
               ,'MSE_val':[0]
               ,'Ratio':[0]
               }    
    for iteration in range(max_iterations):
        if stopped_early:
            break
        
        # Тренировка
        model.train()
        for epoch in range(epochs_per_iteration):
            optimizer.zero_grad()
            
            ed = model.compute_ed(X_train_t, y_train_t)
            ew = model.compute_ew()
            loss = model.compute_regularized_loss(ed, ew, alpha, beta)
            
            loss.backward()
            optimizer.step()
        
        # Вычисление MSE
        model.eval()
        with torch.no_grad():
            train_pred = model(X_train_t)
            val_pred = model(X_val_t)
            
            mse_train = torch.mean((train_pred - y_train_t) ** 2).item()
            mse_val = torch.mean((val_pred - y_val_t) ** 2).item()
        
        if mse_val < best_mse_val:
            best_mse_val = mse_val
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        
        # Обновление α и β с защитой от расходимости
        if (iteration + 1) % 2 == 0 or iteration == 0:
            gamma, n_weights = compute_effective_parameters(model, alpha, beta)
            
            ew_val = model.compute_ew().item()
            ed_val = model.compute_ed(X_train_t, y_train_t).item()
            
            # Новые значения
            if ew_val > 1e-10:
                alpha_new = gamma / (2 * ew_val)
            else:
                alpha_new = alpha
            
            if ed_val > 1e-10:
                beta_new = max(0, (n_data - gamma) / (2 * ed_val))
            else:
                beta_new = beta

            
            # === ЗАЩИТА ОТ РАСХОДИМОСТИ ===
            # Ограничиваем изменение α (не более чем в 2 раза за итерацию)
            # alpha_new = np.clip(alpha_new, alpha_prev / 2, alpha_prev * 2)
            # Ограничиваем α разумным максимумом
            alpha_new = np.clip(alpha_new, 1e-6, 1000.0)
            
            # Ограничиваем изменение β
            # beta_new = np.clip(beta_new, beta_prev / 2, beta_prev * 2)
            beta_new = np.clip(beta_new, 1e-6, 100.0)
            
            alpha = alpha_new
            beta = beta_new
            alpha_prev, beta_prev = alpha, beta
        else:
            gamma, n_weights = compute_effective_parameters(model, alpha, beta)
        
        if verbose:
            ratio = mse_val / (mse_train + 1e-8)
            history['Iter'].append(iteration+1)
            history['alpha'].append(alpha)
            history['beta'].append(beta)
            history['gamma'].append(gamma)
            history['MSE_tr'].append(mse_train)
            history['MSE_val'].append(mse_val)
            history['Ratio'].append(ratio)
        
        # Early stopping
        if mse_val >= 1.9 * mse_train:
            if verbose:
                print(f"    → Early stopping: MSE_val ({mse_val:.4f}) >= 1.9 × MSE_train ({1.9*mse_train:.4f})")
            stopped_early = True
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    display(pd.DataFrame(history))
    return model

## train ensemble

In [ ]:
def train_brann_ensemble(X, y, X_test,
                         input_size=24, n_ensemble=7, test_size=0.25):
    """
    Обучение ансамбля из n_ensemble BRANN с bootstrap
    """
    print(f"\n{'='*80}")
    print(f"ОБУЧЕНИЕ АНСАМБЛЯ ИЗ {n_ensemble} BRANN С BOOTSTRAP")
    print(f"{'='*80}")
    print(f"  Размер обучающей выборки: {len(X)}")
    print(f"  Количество сетей: {n_ensemble}")
    print(f"  Validation size (bootstrap): {test_size*100:.0f}%")
    print(f"{'='*80}\n")
    
    models = []
    predictions_list = []
    
    for i in range(n_ensemble):
        print(f"Сеть #{i+1}/{n_ensemble}:")
        
        # Bootstrap: случайное разбиение train_full на train/val
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=test_size, random_state=i
        )
        
        # Обучение одной BRANN
        model = train_single_brann(
            X_train, y_train, X_val, y_val,
            input_size=input_size,
            max_iterations=50,
            epochs_per_iteration=50,
            alpha_init=0.001,
            beta_init=100,
            verbose=True
        )
        
        models.append(model)
        
        # Предсказание на тестовой выборке
        model.eval()
        with torch.no_grad():
            X_test_t = torch.FloatTensor(X_test)
            pred = model(X_test_t).cpu().numpy().flatten()
            predictions_list.append(pred)
        
        print(f"  ✓ Сеть #{i+1} обучена\n")
    
    # Усреднение предсказаний ансамбля
    predictions_array = np.array(predictions_list)  # Shape: (7, n_test)
    final_predictions = np.mean(predictions_array, axis=0)
    
    # Стандартное отклонение предсказаний (неопределенность)
    pred_std = np.std(predictions_array, axis=0)
    
    return models, final_predictions, pred_std

In [ ]:
def train_single_brann_debug(X_train, y_train, X_val, y_val, 
                             input_size=24, max_iterations=100, epochs_per_iteration=50,
                             alpha_init=0.0, beta_init=1.0, learning_rate=0.01,
                             verbose=True):
    """
    Упрощенное обучение БЕЗ регуляризации для проверки что сеть вообще обучается
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = BRANN(input_size).to(device)
    
    X_train_t = torch.FloatTensor(X_train).to(device)
    y_train_t = torch.FloatTensor(y_train).view(-1, 1).to(device)
    X_val_t = torch.FloatTensor(X_val).to(device)
    y_val_t = torch.FloatTensor(y_val).view(-1, 1).to(device)
    
    # Начинаем БЕЗ регуляризации (alpha=0)
    alpha = alpha_init
    beta = beta_init
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    if verbose:
        print(f"    {'Iter':>4} | {'Loss':>10} | {'MSE_tr':>8} | {'MSE_val':>8}")
        print(f"    {'-'*4}|{'-'*12}|{'-'*10}|{'-'*10}")
    
    for iteration in range(max_iterations):
        model.train()
        for epoch in range(epochs_per_iteration):
            optimizer.zero_grad()
            
            # ТОЛЬКО MSE, без регуляризации!
            predictions = model(X_train_t)
            loss = torch.mean((predictions - y_train_t) ** 2)
            
            loss.backward()
            
            # Проверка градиентов
            if iteration == 0 and epoch == 0 and verbose:
                total_grad_norm = 0
                for param in model.parameters():
                    if param.grad is not None:
                        total_grad_norm += param.grad.norm().item()
                print(f"    Total grad norm: {total_grad_norm:.6f}")
            
            optimizer.step()
        
        # Оценка
        model.eval()
        with torch.no_grad():
            train_pred = model(X_train_t)
            val_pred = model(X_val_t)
            
            mse_train = torch.mean((train_pred - y_train_t) ** 2).item()
            mse_val = torch.mean((val_pred - y_val_t) ** 2).item()
        
        if verbose and (iteration % 10 == 0 or iteration < 5):
            print(f"    {iteration+1:>4} | {loss.item():>10.6f} | {mse_train:>8.6f} | {mse_val:>8.6f}")
    
    return model

In [ ]:
print("\nТЕСТОВОЕ ОБУЧЕНИЕ (без регуляризации):")
model_test = train_single_brann_debug(
    X_scaled, y_scaled, 
    input_size=m+1,
    max_iterations=50,
    epochs_per_iteration=100,  # Много эпох
    learning_rate=0.1,  # Большой LR
    verbose=True
)

## main

In [271]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X_temp).astype("float32") # ???
y_scaled = scaler_y.fit_transform(y_temp).astype("float32").reshape(-1,1)

X_test_scaled = scaler_X.transform(X_test).astype("float32")
# y_test_scaled = scaler_y.transform(y_test).astype("float32").reshape(-1,1)

# 2. Ансамбль из 7 сетей (BRANN / Bootstrap procedure)
# Текст: "The modeling ensemble consists of seven networks that act in parallel."
n_ensemble = 7


In [272]:


models, predictions_scaled, pred_std = train_brann_ensemble(
        X_scaled, y_scaled, X_test_scaled,
        input_size=27,
        n_ensemble=7,
        test_size=0.25  # 25% для validation при bootstrap
    )



ОБУЧЕНИЕ АНСАМБЛЯ ИЗ 7 BRANN С BOOTSTRAP
  Размер обучающей выборки: 2929
  Количество сетей: 7
  Validation size (bootstrap): 25%

Сеть #1/7:


,Iter,alpha,beta,gamma,MSE_tr,MSE_val,Ratio
0,0,0.001000,100.000000,0.000000,0.000000,0.000000,0.000000
1,1,1.656915,1.313156,377.998110,0.315221,0.320584,1.017013
2,2,1.210398,1.406282,272.562472,0.311418,0.317909,1.020843
3,3,1.210398,1.406282,290.574269,0.309521,0.317156,1.024667
4,4,1.516995,1.404963,290.574269,0.308791,0.316620,1.025352
5,5,1.516995,1.404963,279.876753,0.308430,0.316101,1.024870
6,6,1.715973,1.415898,279.876753,0.308127,0.315044,1.022449
7,7,1.715973,1.415898,274.445633,0.307937,0.314523,1.021388
8,8,1.982827,1.421776,274.445633,0.307722,0.313894,1.020057
9,9,1.982827,1.421776,267.927173,0.307545,0.313225,1.018469


  ✓ Сеть #1 обучена

Сеть #2/7:


,Iter,alpha,beta,gamma,MSE_tr,MSE_val,Ratio
0,0,0.001000,100.000000,0.000000,0.000000,0.000000,0.000000
1,1,1.449254,1.450350,377.998110,0.285403,0.390977,1.369909
2,2,1.115658,1.549573,283.535700,0.281008,0.400078,1.423725
3,3,1.115658,1.549573,298.885137,0.278988,0.399040,1.430312
4,4,1.410979,1.554569,298.885137,0.277857,0.398028,1.432492
5,5,1.410979,1.554569,288.075609,0.276843,0.395522,1.428689
6,6,1.645359,1.575221,288.075609,0.275776,0.392061,1.421664
7,7,1.645359,1.575221,281.441977,0.274776,0.391007,1.423001
8,8,1.912761,1.591505,281.441977,0.273904,0.391446,1.429135
9,9,1.912761,1.591505,274.836631,0.273378,0.389309,1.424066


  ✓ Сеть #2 обучена

Сеть #3/7:


...

  ✓ Сеть #5 обучена

Сеть #6/7:


,Iter,alpha,beta,gamma,MSE_tr,MSE_val,Ratio
0,0,0.001000,100.000000,0.000000,0.000000,0.000000,0.000000
1,1,1.013672,1.382704,377.998110,0.299366,0.348095,1.162773
2,2,0.921402,1.458894,298.052609,0.296209,0.350297,1.182602
3,3,0.921402,1.458894,304.838934,0.293894,0.344566,1.172418
4,4,1.195942,1.470659,304.838934,0.292789,0.342456,1.169634
5,5,1.195942,1.470659,293.235530,0.291899,0.341711,1.170648
6,6,1.404631,1.488105,293.235530,0.291131,0.340622,1.169995
7,7,1.404631,1.488105,286.226923,0.290490,0.339516,1.168769
8,8,1.649656,1.499764,286.226923,0.289932,0.338366,1.167052
9,9,1.649656,1.499764,279.002427,0.289520,0.337957,1.167304


  ✓ Сеть #6 обучена

Сеть #7/7:


,Iter,alpha,beta,gamma,MSE_tr,MSE_val,Ratio
0,0,0.001000,100.000000,0.000000,0.000000,0.000000,0.000000
1,1,1.506914,1.247788,377.998110,0.331735,0.279604,0.842855
2,2,1.083844,1.330797,274.610658,0.328731,0.276408,0.840831
3,3,1.083844,1.330797,293.164817,0.326323,0.272417,0.834808
4,4,1.346610,1.332336,293.164817,0.325181,0.271598,0.835221
5,5,1.346610,1.332336,282.996485,0.324272,0.270842,0.835231
6,6,1.539108,1.347338,282.996485,0.323278,0.269615,0.834001
7,7,1.539108,1.347338,277.221627,0.322381,0.268755,0.833656
8,8,1.791760,1.359449,277.221627,0.321366,0.267690,0.832975
9,9,1.791760,1.359449,270.535650,0.320418,0.267026,0.833368


  ✓ Сеть #7 обучена



In [273]:
y_pred_mean = np.asarray(predictions_scaled).reshape(-1, 1)
y_test = np.asarray(y_test).ravel()

y_pred = scaler_y.inverse_transform(y_pred_mean)
# Обратная логарифмическая трансформация (формула 4)
sigma_pred = sigma_from_y(y_pred)
sigma_real = sigma_from_y(y_test)
sigma_pred_tensor = torch.FloatTensor(sigma_pred).view(-1, 1)
sigma_real_tensor = torch.FloatTensor(sigma_real).view(-1, 1)

# Расчет ошибок
rmse_abs, rmse_rel, mape, r2 = calculate_metrics_np(sigma_pred, sigma_real)

print("\n" + "="*80)
print("РЕЗУЛЬТАТЫ МОДЕЛИРОВАНИЯ")
print("="*80)
print(f"  RMSE (абсолютная):    {rmse_abs:>10.2f} MPa")
print(f"  RMSE (относительная): {rmse_rel:>10.4f} ({rmse_rel*100:.2f}%)")
print(f"  MAPE:                 {mape:>10.2f} %")
print(f"  R²:                   {r2:>10.4f}")
print(f"  Диапазон предсказаний: [{sigma_pred.min():.1f}, {sigma_pred.max():.1f}] MPa")
print("="*80)



РЕЗУЛЬТАТЫ МОДЕЛИРОВАНИЯ
  RMSE (абсолютная):        301.28 MPa
  RMSE (относительная):     2.6771 (267.71%)
  MAPE:                     121.05 %
  R²:                    -893.7502
  Диапазон предсказаний: [32.3, 1231.1] MPa
